In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [2]:
if torch.cuda.is_available():
    # Print the name of the GPU
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    # Print the number of available CUDA devices
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
else:
    print("No GPU available. Training will run on CPU.")


No GPU available. Training will run on CPU.


In [3]:
batch_size = 2
num_decoder_tokens = 3
num_encoder_tokens = 5
d_model = 8
num_heads = 2
head_dim = 4

In [4]:
w_q = nn.Linear(d_model, d_model, bias=False)
w_k = nn.Linear(d_model, d_model, bias=False)
w_v = nn.Linear(d_model, d_model, bias=False)
w_o = nn.Linear(d_model, d_model, bias=False)

In [5]:
dummy_encoder_input = torch.rand(batch_size, num_encoder_tokens, d_model)
dummy_decoder_input = torch.rand(batch_size, num_decoder_tokens, d_model)
print(dummy_encoder_input.shape)
print(dummy_decoder_input.shape)
print("(batch_size, num_tokens, embedding_dimension)")
dummy_decoder_input

torch.Size([2, 5, 8])
torch.Size([2, 3, 8])
(batch_size, num_tokens, embedding_dimension)


tensor([[[0.1603, 0.9182, 0.2920, 0.7353, 0.5995, 0.0560, 0.8882, 0.0971],
         [0.0716, 0.3145, 0.3016, 0.9070, 0.8880, 0.6595, 0.7045, 0.4800],
         [0.4374, 0.0763, 0.3769, 0.5457, 0.5300, 0.2589, 0.3298, 0.8827]],

        [[0.3204, 0.0815, 0.7418, 0.7191, 0.1187, 0.0050, 0.6742, 0.8560],
         [0.2361, 0.6579, 0.5616, 0.2514, 0.1283, 0.8148, 0.5117, 0.8618],
         [0.2681, 0.0224, 0.9726, 0.2055, 0.3238, 0.2549, 0.6264, 0.3795]]])

In [6]:
q = w_q(dummy_decoder_input)
k = w_k(dummy_encoder_input)
v = w_v(dummy_encoder_input)

In [7]:
print(q.shape)
print(k.shape)
print("(batch_size, num_tokens, key_dimension)")
q

torch.Size([2, 3, 8])
torch.Size([2, 5, 8])
(batch_size, num_tokens, key_dimension)


tensor([[[-0.1257,  0.0639, -0.0481, -0.0565,  0.5758,  0.3512, -0.2378,
          -0.2506],
         [-0.4188,  0.0628, -0.0056, -0.1774,  0.5418,  0.3020, -0.6253,
          -0.4089],
         [-0.5633, -0.1482, -0.0212, -0.0433,  0.2398,  0.0877, -0.4661,
          -0.4184]],

        [[-0.6516, -0.1021,  0.0457,  0.1110,  0.3613, -0.0108, -0.4608,
          -0.3693],
         [-0.6043, -0.1872,  0.0279, -0.2247,  0.2479, -0.1302, -0.6421,
          -0.4631],
         [-0.5043,  0.0889, -0.0990,  0.0126,  0.2851,  0.0168, -0.4223,
          -0.1658]]], grad_fn=<ViewBackward0>)

In [8]:
q_head = q.view(batch_size, num_decoder_tokens, num_heads, head_dim)
k_head = k.view(batch_size, num_encoder_tokens, num_heads, head_dim)
v_head = v.view(batch_size, num_encoder_tokens, num_heads, head_dim)
print(q_head.shape)
print(k_head.shape)
q_head

torch.Size([2, 3, 2, 4])
torch.Size([2, 5, 2, 4])


tensor([[[[-0.1257,  0.0639, -0.0481, -0.0565],
          [ 0.5758,  0.3512, -0.2378, -0.2506]],

         [[-0.4188,  0.0628, -0.0056, -0.1774],
          [ 0.5418,  0.3020, -0.6253, -0.4089]],

         [[-0.5633, -0.1482, -0.0212, -0.0433],
          [ 0.2398,  0.0877, -0.4661, -0.4184]]],


        [[[-0.6516, -0.1021,  0.0457,  0.1110],
          [ 0.3613, -0.0108, -0.4608, -0.3693]],

         [[-0.6043, -0.1872,  0.0279, -0.2247],
          [ 0.2479, -0.1302, -0.6421, -0.4631]],

         [[-0.5043,  0.0889, -0.0990,  0.0126],
          [ 0.2851,  0.0168, -0.4223, -0.1658]]]], grad_fn=<ViewBackward0>)

In [9]:
q_head = q_head.transpose(1, 2)
k_head = k_head.transpose(1, 2)
v_head = v_head.transpose(1, 2)
print(q_head.shape)
print(k_head.shape)
print("(batch_size, num_heads, num_tokens, head_dim)")
q_head

torch.Size([2, 2, 3, 4])
torch.Size([2, 2, 5, 4])
(batch_size, num_heads, num_tokens, head_dim)


tensor([[[[-0.1257,  0.0639, -0.0481, -0.0565],
          [-0.4188,  0.0628, -0.0056, -0.1774],
          [-0.5633, -0.1482, -0.0212, -0.0433]],

         [[ 0.5758,  0.3512, -0.2378, -0.2506],
          [ 0.5418,  0.3020, -0.6253, -0.4089],
          [ 0.2398,  0.0877, -0.4661, -0.4184]]],


        [[[-0.6516, -0.1021,  0.0457,  0.1110],
          [-0.6043, -0.1872,  0.0279, -0.2247],
          [-0.5043,  0.0889, -0.0990,  0.0126]],

         [[ 0.3613, -0.0108, -0.4608, -0.3693],
          [ 0.2479, -0.1302, -0.6421, -0.4631],
          [ 0.2851,  0.0168, -0.4223, -0.1658]]]],
       grad_fn=<TransposeBackward0>)

In [10]:
k_t = k_head.transpose(-2, -1)
print(k_t.shape)
k_t

torch.Size([2, 2, 4, 5])


tensor([[[[-0.1186, -0.2941, -0.0157, -0.0220,  0.1258],
          [-0.3489, -0.3301, -0.1498, -0.3869, -0.5884],
          [ 0.6674,  0.5037,  0.8009,  0.8309,  0.7870],
          [-0.4113, -0.4211, -0.3552, -0.3571, -0.2919]],

         [[-0.1705, -0.4909, -0.1066, -0.2822, -0.0435],
          [-0.0299, -0.1245, -0.2156, -0.1763, -0.0297],
          [-0.1591,  0.0061, -0.2892, -0.3510, -0.0042],
          [-0.1752, -0.0625, -0.2836,  0.1186,  0.1231]]],


        [[[-0.1147, -0.1992, -0.2295,  0.0646, -0.1198],
          [-0.3213, -0.2461, -0.4507, -0.4737, -0.1228],
          [ 0.6797,  0.6581,  0.7217,  0.6653,  0.6308],
          [-0.4098, -0.3802, -0.4637, -0.3588, -0.3677]],

         [[-0.2987, -0.0998, -0.4643, -0.1564, -0.1431],
          [-0.0968, -0.0045,  0.0135,  0.2849,  0.0503],
          [-0.1651,  0.2916,  0.0407, -0.1060, -0.0595],
          [-0.0437, -0.1172,  0.3636,  0.1768, -0.2117]]]],
       grad_fn=<TransposeBackward0>)

In [11]:
sim1 = (q_head @ k_t)/math.sqrt(head_dim)
print(sim1.shape)
print("(batch, num_heads, num_queries, num_keys)")
sim1
# Attention Scores Matrix

torch.Size([2, 2, 3, 5])
(batch, num_heads, num_queries, num_keys)


tensor([[[[-0.0081,  0.0077, -0.0130, -0.0209, -0.0374],
          [ 0.0485,  0.0872,  0.0278,  0.0218, -0.0211],
          [ 0.0611,  0.1111,  0.0147,  0.0338,  0.0061]],

         [[-0.0135, -0.1561,  0.0014, -0.0853, -0.0327],
          [ 0.0348, -0.1409,  0.0870, -0.0176, -0.0401],
          [ 0.0520, -0.0527,  0.1045,  0.0154, -0.0313]]],


        [[[ 0.0466,  0.0714,  0.0885, -0.0016,  0.0393],
          [ 0.1203,  0.1351,  0.1737,  0.0744,  0.0978],
          [-0.0216,  0.0043, -0.0008, -0.0725, -0.0088]],

         [[-0.0073, -0.0636, -0.1605, -0.0380,  0.0267],
          [ 0.0324, -0.0786, -0.1557, -0.0448,  0.0471],
          [-0.0049, -0.0661, -0.1048, -0.0122,  0.0101]]]],
       grad_fn=<DivBackward0>)

In [12]:
sim2 = F.softmax(sim1, dim=-1)
print(sim2.shape)
print("(num_batches, num_heads, num_queries, num_keys)")
sim2
# Attention Weights matrix
# Basically building a neural network for value vectors

torch.Size([2, 2, 3, 5])
(num_batches, num_heads, num_queries, num_keys)


tensor([[[[0.2012, 0.2044, 0.2002, 0.1987, 0.1954],
          [0.2030, 0.2110, 0.1989, 0.1977, 0.1894],
          [0.2030, 0.2134, 0.1938, 0.1976, 0.1922]],

         [[0.2086, 0.1809, 0.2117, 0.1941, 0.2046],
          [0.2097, 0.1759, 0.2209, 0.1990, 0.1945],
          [0.2067, 0.1861, 0.2178, 0.1992, 0.1902]]],


        [[[0.1994, 0.2045, 0.2080, 0.1901, 0.1980],
          [0.1999, 0.2029, 0.2109, 0.1909, 0.1954],
          [0.1996, 0.2048, 0.2038, 0.1897, 0.2022]],

         [[0.2080, 0.1966, 0.1785, 0.2017, 0.2152],
          [0.2144, 0.1919, 0.1776, 0.1985, 0.2176],
          [0.2060, 0.1938, 0.1865, 0.2045, 0.2092]]]],
       grad_fn=<SoftmaxBackward0>)

In [13]:
print(v_head.shape)
v_head # Value vectors

torch.Size([2, 2, 5, 4])


tensor([[[[ 0.5421,  0.0009,  0.3355,  0.1260],
          [ 0.6173,  0.1604,  0.2093,  0.1240],
          [ 0.2585, -0.0459,  0.5316,  0.1159],
          [ 0.3063,  0.2680,  0.6441, -0.1198],
          [ 0.5406,  0.0551,  0.4515, -0.0761]],

         [[-0.6295, -0.1931,  0.3597,  0.2050],
          [-0.6348, -0.2771,  0.2742,  0.3515],
          [-0.4338, -0.3985,  0.2148, -0.1588],
          [-0.3197, -0.0682,  0.5765, -0.0601],
          [-0.4614,  0.0490,  0.5433,  0.0620]]],


        [[[ 0.3019,  0.1059,  0.4110, -0.1018],
          [ 0.6304, -0.3344,  0.0393,  0.1138],
          [ 0.5187,  0.2075,  0.2996, -0.2417],
          [ 0.5124,  0.0803,  0.2917, -0.0702],
          [ 0.4182, -0.1940,  0.1903,  0.1526]],

         [[-0.2784, -0.2964,  0.4780, -0.1823],
          [-0.5728, -0.2402,  0.3284,  0.1476],
          [-0.2908,  0.0264,  0.7118,  0.0639],
          [-0.5008,  0.1164,  0.3592,  0.1427],
          [-0.5479, -0.3024,  0.1155,  0.0532]]]],
       grad_fn=<TransposeBack

In [14]:
sim3 = sim2 @ v_head
print(sim3.shape)
sim3 
# Applying Attention Weights to value vectors
# Essentially, passing value vectors through a neural network

torch.Size([2, 2, 3, 4])


tensor([[[[ 0.4536,  0.0878,  0.4329,  0.0352],
          [ 0.4547,  0.0883,  0.4308,  0.0367],
          [ 0.4563,  0.0891,  0.4298,  0.0362]],

         [[-0.4945, -0.1780,  0.3932,  0.0737],
          [-0.4928, -0.1813,  0.3915,  0.0698],
          [-0.4942, -0.1825,  0.3903,  0.0730]]],


        [[[ 0.4772, -0.0272,  0.2455, -0.0304],
          [ 0.4772, -0.0255,  0.2462, -0.0318],
          [ 0.4768, -0.0291,  0.2449, -0.0287]],

         [[-0.4414, -0.1458,  0.3883,  0.0427],
          [-0.4399, -0.1476,  0.3884,  0.0405],
          [-0.4396, -0.1421,  0.3925,  0.0433]]]],
       grad_fn=<UnsafeViewBackward0>)

In [15]:
sim3 = sim3.transpose(1, 2).contiguous()
sim3.shape

torch.Size([2, 3, 2, 4])

In [16]:
sim3 = sim3.view(batch_size, num_decoder_tokens, d_model)
print(sim3.shape)
sim3

torch.Size([2, 3, 8])


tensor([[[ 0.4536,  0.0878,  0.4329,  0.0352, -0.4945, -0.1780,  0.3932,
           0.0737],
         [ 0.4547,  0.0883,  0.4308,  0.0367, -0.4928, -0.1813,  0.3915,
           0.0698],
         [ 0.4563,  0.0891,  0.4298,  0.0362, -0.4942, -0.1825,  0.3903,
           0.0730]],

        [[ 0.4772, -0.0272,  0.2455, -0.0304, -0.4414, -0.1458,  0.3883,
           0.0427],
         [ 0.4772, -0.0255,  0.2462, -0.0318, -0.4399, -0.1476,  0.3884,
           0.0405],
         [ 0.4768, -0.0291,  0.2449, -0.0287, -0.4396, -0.1421,  0.3925,
           0.0433]]], grad_fn=<ViewBackward0>)

In [17]:
output = w_o(sim3)
# Projecting from key_dimension back to d_model (up projection)
print(output.shape)
output
# We have output tensor of the same dimension as our decoder input. Ready to be passed on to the next layer.

torch.Size([2, 3, 8])


tensor([[[ 0.6303, -0.1382,  0.0183,  0.2174, -0.2673, -0.1365,  0.0204,
           0.0258],
         [ 0.6306, -0.1365,  0.0192,  0.2161, -0.2677, -0.1387,  0.0222,
           0.0260],
         [ 0.6301, -0.1368,  0.0194,  0.2161, -0.2689, -0.1379,  0.0238,
           0.0245]],

        [[ 0.5978, -0.1120, -0.0488,  0.1664, -0.1722, -0.1187,  0.0429,
           0.0604],
         [ 0.5979, -0.1109, -0.0482,  0.1659, -0.1724, -0.1208,  0.0428,
           0.0606],
         [ 0.5981, -0.1133, -0.0492,  0.1686, -0.1707, -0.1189,  0.0417,
           0.0609]]], grad_fn=<ViewBackward0>)

In [18]:
class CrossMultiHeadAttention(nn.Module):
    """
    Compute Scaled Dot Product Attention using Multiple Heads
    """

    def __init__(self, d_model:int, num_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.w_q = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_k = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_v = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_o = nn.Linear(self.d_model, self.d_model, bias=False)

        # Checking if d_model is divisible by num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.head_dim = d_model // num_heads # Dimension of query, key and value vectors within each head

    def forward(self, encoder_input: torch.Tensor, decoder_input: torch.Tensor) -> torch.Tensor:
        
        # Checking if input has correct embedding dimension
        if encoder_input.shape[-1] != self.d_model:
            raise ValueError(f"Expected encoder input feature dimension to be d_model ({self.d_model}), but got {encoder_input.shape[-1]}")

        if decoder_input.shape[-1] != self.d_model:
            raise ValueError(f"Expected decoder input feature dimension to be d_model ({self.d_model}), but got {decoder_input.shape[-1]}")

        # Input Dimension: (batch_size, num_tokens, d_model)
        batch_size = decoder_input.shape[0]
        encoder_num_tokens = encoder_input.shape[1]
        decoder_num_tokens = decoder_input.shape[1]

        # Creating query, key and value vectors from input embeddings. 
        # These vectors contain queries from all the heads combined.
        q = self.w_q(decoder_input)
        k = self.w_k(encoder_input)
        v = self.w_v(encoder_input)
        # Dimension: (batch_sizee, num_tokens, d_model)

        # Splitting our vectors into heads by reshaping our tensors.
        q_head = q.view(batch_size, decoder_num_tokens, self.num_heads, self.head_dim)
        k_head = k.view(batch_size, encoder_num_tokens, self.num_heads, self.head_dim)
        v_head = v.view(batch_size, encoder_num_tokens, self.num_heads, self.head_dim)
        # Dimension: (batch_size, num_tokens, num_heads, head_dimension)

        # Swapping (num_tokens) and (num_heads) dimensions.
        # We do this because we need (num_tokens) and (head_dimension) at the end.
        # This is because we need to perform matrix operations on the last two dimensions
        q_head = q_head.transpose(1, 2)
        k_head = k_head.transpose(1, 2)
        v_head = v_head.transpose(1, 2)
        # Dimension: (batch_size, num_heads, num_tokens, head_dimension)

        # Getting a transpose of key vectors for matrix multiplication 
        k_t = k_head.transpose(-2, -1)
        sim1 = (q_head @ k_t)/math.sqrt(self.head_dim) # Scaled dot product of query and key vectors
        # sim1 is the matrix of attention scores for each token
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Performing softmax along the keys dimension to get attention scores for each key
        sim2 = F.softmax(sim1, dim=-1)
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Weighted sum of attention weights and value vectors
        sim3 = sim2 @ v_head
        # Dimension: (batch_size, num_heads, num_queries, head_dimension)

        # Swapping (num_heads) and (num_queries) dimension back so that we can recombine all the heads
        sim3 = sim3.transpose(1, 2).contiguous()

        # Recombining all the heads
        sim3 = sim3.view(batch_size, decoder_num_tokens, self.d_model)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        # Passing all of our heads into a final layer so that all heads can interact.
        output = self.w_o(sim3)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        return output

In [19]:
dummy_input = torch.rand(2, 3, 8)
dummy_input

tensor([[[0.1338, 0.1965, 0.7858, 0.7675, 0.0278, 0.6886, 0.5254, 0.4112],
         [0.3181, 0.9575, 0.2166, 0.8274, 0.4443, 0.3731, 0.1022, 0.8327],
         [0.6064, 0.2591, 0.8907, 0.4084, 0.6161, 0.6322, 0.3418, 0.8041]],

        [[0.6067, 0.2042, 0.3229, 0.7436, 0.2632, 0.1238, 0.4839, 0.8053],
         [0.9632, 0.6583, 0.5478, 0.6403, 0.3653, 0.1184, 0.2642, 0.1302],
         [0.2837, 0.0924, 0.8965, 0.3547, 0.5319, 0.4216, 0.7313, 0.7537]]])

In [20]:
mha_layer = CrossMultiHeadAttention(d_model=8, num_heads=2)
mha_layer

CrossMultiHeadAttention(
  (w_q): Linear(in_features=8, out_features=8, bias=True)
  (w_k): Linear(in_features=8, out_features=8, bias=True)
  (w_v): Linear(in_features=8, out_features=8, bias=True)
  (w_o): Linear(in_features=8, out_features=8, bias=True)
)

In [21]:
output = mha_layer(dummy_encoder_input, dummy_decoder_input)
output

tensor([[[ 0.2918,  0.3201,  0.1722, -0.2486, -0.4305, -0.5479, -0.2025,
           0.3063],
         [ 0.2909,  0.3208,  0.1709, -0.2485, -0.4293, -0.5494, -0.2027,
           0.3067],
         [ 0.2906,  0.3192,  0.1711, -0.2481, -0.4269, -0.5482, -0.2028,
           0.3046]],

        [[ 0.3200,  0.2437,  0.1682, -0.2491, -0.2502, -0.5562, -0.1387,
           0.2102],
         [ 0.3191,  0.2443,  0.1667, -0.2493, -0.2482, -0.5572, -0.1398,
           0.2090],
         [ 0.3198,  0.2431,  0.1692, -0.2478, -0.2496, -0.5573, -0.1374,
           0.2100]]], grad_fn=<ViewBackward0>)